# 对话缓冲记忆（Conversation Buffer Memory）

> **最基本的智能体记忆形式：把整段对话一字不落地存下来。**

每个 AI 智能体都需要记忆。没有记忆，每一轮都是从零开始。智能体无法引用你一分钟前说过的话，更别提连贯地进行多轮对话了。

想象一台从不暂停的录音机。它录下每一个字，智能体要回复时，就把整盘带子重放一遍。这就是**对话缓冲记忆**。它是所有其他记忆技术赖以构建的地基。思路很直接：

1. 维护一个有序的消息列表。
2. 每一轮都把*整个*列表发给 LLM。
3. 把回复追加进去。如此往复。

本 notebook 用 **deepagents**（LangChain 团队开源的深度智能体框架）搭配任意 **OpenAI 兼容端点**，从零构建缓冲记忆。你会用 **matplotlib** 可视化线性 token 增长问题，还会亲眼看到：不带缓冲的 deepagents 每一轮都在「失忆」。

**读完你将理解：**
- 为什么缓冲记忆是任何智能体的默认起点。
- 消息如何累积，成本和延迟为何随轮次线性增长。
- 这种方法何时失效，下一步该怎么做。


## 关键概念

- **消息列表**：由 `{role, content}` 字典构成的有序列表。角色通常是 `user` 和 `assistant`（外加一个可选的 `system` 提示）。
- **角色交替**：LLM 依赖正确的角色顺序来判断谁说了什么。
- **上下文注入**：每次 API 调用都把完整消息列表作为提示词传入。「上下文」指模型在回答前读到的文本。
- **token 计数**：每条消息都消耗 token（模型内部使用的词片段）。累计总量决定了成本，也决定了你会不会撞上上下文窗口的天花板。
- **线性增长**：每个新轮次都会增加 token，而之前*所有* token 也会被重发。所以累计 token 用量随轮次呈 **O(n²)** 增长，其中 *n* 是轮数。这意味着轮数翻倍，总成本大约翻四倍。


## Architecture

<p align="center">
  <img src="../../images/diagrams/01_conversation_buffer_memory.svg" alt="对话缓冲记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
sequenceDiagram
    participant U as User
    participant B as Buffer Memory
    participant L as LLM

    U->>B: "Hi, I'm Alice"
    B->>B: Append {role: user, content: "Hi, I'm Alice"}
    B->>L: [msg1]
    L-->>B: "Hello Alice! How can I help?"
    B->>B: Append {role: assistant, content: "Hello Alice!..."}

    U->>B: "What's my name?"
    B->>B: Append {role: user, content: "What's my name?"}
    B->>L: [msg1, msg2, msg3]
    L-->>B: "Your name is Alice."
    B->>B: Append {role: assistant, content: "Your name is Alice."}

    Note over B: Buffer grows with every turn.<br/>All messages re-sent each call.
```

</details>

## 环境准备

安装依赖并配置 API 访问。本 notebook 使用任意 OpenAI 兼容端点（官方 OpenAI、代理服务或私有部署都行）。


In [ ]:
%pip install -q deepagents python-dotenv matplotlib tiktoken langchain-openai

导入标准库工具，从 `.env` 加载 `LLM_API_KEY` / `LLM_BASE_URL` / `LLM_MODEL_NAME`，构建一个 `ChatOpenAI` 模型实例。deepagents 不绑定某一家模型，任何 OpenAI 兼容端点都能接。`to_text` 是个小工具：有的端点把回复返回成内容块列表而不是字符串，用它统一转成纯文本。


In [ ]:
import os
import json
import copy
from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)


def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )

## 实现

我们将构建一个极简的 `ConversationBufferMemory` 类，它：
1. 把消息存在一个普通的 Python 列表里。
2. 内部用 `create_deep_agent` 创建一个 deepagents 智能体。
3. 每次调用都把完整历史交给智能体。
4. 记录每轮的 token 用量，供之后可视化。

关键认知：deepagents 的 `invoke` 是**无状态**的。每次调用它只看到你传进去的消息。不传历史，它就什么都不记得。缓冲记忆要做的，就是把这份历史一直带上。


In [ ]:
class ConversationBufferMemory:
    """基于 deepagents 的极简对话缓冲记忆。"""

    def __init__(self, model, system_prompt: str | None = None):
        # deepagents 智能体本身无状态，记忆全靠下面这个列表
        self.agent = create_deep_agent(model=model, system_prompt=system_prompt)
        self.messages: list[dict] = []          # 核心数据结构：消息缓冲区
        self.turn_token_usage: list[dict] = []  # 记录每轮 token 用量，供可视化

    # ── 消息辅助方法 ──────────────────────────────────────────────
    def add_user_message(self, content: str) -> None:
        self.messages.append({"role": "user", "content": content})

    def add_assistant_message(self, content: str) -> None:
        self.messages.append({"role": "assistant", "content": content})

    # ── 聊天 ─────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:
        """发送消息并获取回复。每次都会发送*整个*缓冲区。"""
        self.add_user_message(user_input)
        result = self.agent.invoke({"messages": self.messages})  # <-- 每次都传完整历史
        reply = result["messages"][-1]
        reply_text = to_text(reply.content)
        self.add_assistant_message(reply_text)

        # 记录这一轮的 token 用量：优先用接口返回值，读不到就用 tiktoken 估算
        usage = getattr(reply, "usage_metadata", None) or {}
        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": int(usage.get("input_tokens") or self._estimate_tokens(self.messages)),
            "output_tokens": int(usage.get("output_tokens") or self._estimate_tokens(reply_text)),
        })
        return reply_text

    # ── 实用方法 ─────────────────────────────────────────────────
    @staticmethod
    def _estimate_tokens(text) -> int:
        """tiktoken 粗略估算，仅在端点不返回 usage 时使用。"""
        import tiktoken
        if not isinstance(text, str):
            text = json.dumps(text, ensure_ascii=False)
        return len(tiktoken.get_encoding("cl100k_base").encode(text))

    def get_history(self) -> list[dict]:
        return copy.deepcopy(self.messages)

    def clear(self) -> None:
        self.messages.clear()
        self.turn_token_usage.clear()

    def __len__(self) -> int:
        return len(self.messages)

    def __repr__(self) -> str:
        return f"ConversationBufferMemory({len(self.messages)} messages)"

类已经完整：`chat` 方法是缓冲记忆干活的地方。它追加用户消息，把**完整**消息列表交给 deepagents，再从返回状态里取最后一条 AI 回复。注意 `self.messages`（完整列表）每次都会传入，这正是缓冲记忆的定义性特征。

token 用量优先读接口返回的 usage 数据；有些端点不返回，就用 tiktoken 粗略估算兜底。先用一次真实调用自检，确认记忆的读写都通了。


In [ ]:
# 自检：一次 chat 之后，缓冲区应恰好有 user + assistant 两条消息
probe = ConversationBufferMemory(model=llm, system_prompt="Reply with OK.")
probe.chat("Say OK.")
assert len(probe) == 2, "缓冲区应包含两条消息"
assert probe.messages[0]["role"] == "user" and probe.messages[1]["role"] == "assistant"
print(probe, "✓ 记忆读写正常")
print(f"本轮实测 token: {probe.turn_token_usage[0]}")

## 运行示例

一段简短的多轮对话，展示缓冲记忆的实际效果。智能体什么都记得，因为它每一轮都重读整个历史。


In [ ]:
memory = ConversationBufferMemory(
    model=llm,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

# 多轮对话
exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer.",
    "I'm working on a project about agent memory. Any tips?",
    "What's my name and what do I do?",  # 回忆测试
]

for msg in exchanges:
    print(f"👤 User:  {msg}")
    reply = memory.chat(msg)
    print(f"🤖 Agent: {reply}\n")

print(f"Buffer now contains {len(memory)} messages.")

查看原始消息缓冲区。这正是 LLM 每次调用收到的内容。


In [ ]:
# 看一眼原始消息缓冲区
for i, msg in enumerate(memory.get_history()):
    role_label = "USER" if msg["role"] == "user" else "ASST"
    # 截断过长消息便于展示
    preview = msg["content"][:80] + ("..." if len(msg["content"]) > 80 else "")
    print(f"  [{i}] {role_label}: {preview}")

## 线性 token 增长问题

缓冲记忆下，**每一轮都会重发之前所有消息**。情况是这样的：

| 轮次 | 发给 LLM 的 token |
|------|----------------------|
| 1 | 只有第一条消息 |
| 2 | 消息 1-3 |
| 3 | 消息 1-5 |
| *n* | 消息 1-(2n-1) |

每轮输入 token 随对话长度**线性**增长。整场对话的**累计** token 用量则**二次方**增长。我们用一段更长的对话和一张图来证明。

一个容易让你意外的数字：第 1 轮就有约 2,000 输入 token。这是 deepagents 框架的固定开销：它自带系统提示词和内置工具定义（文件读写、子智能体等），每次调用都会带上。这个底数不随对话增长，真正重要的是曲线的**斜率**：每多一轮，输入 token 就多一段。


In [ ]:
# 跑一段更长的对话，观察增长曲线
memory_demo = ConversationBufferMemory(
    model=llm,
    system_prompt="You are a helpful assistant. Keep answers to 1-2 sentences.",
)

demo_messages = [
    "Hello! I'm Bob.",
    "I live in San Francisco.",
    "I have a golden retriever named Max.",
    "My favorite programming language is Python.",
    "I work at a startup that builds AI tools.",
    "We're building a chatbot with memory capabilities.",
    "Our team has 12 engineers.",
    "We use GPT as our primary LLM.",
    "What do you know about me so far?",
    "Summarize everything you remember about me in a list.",
]

for msg in demo_messages:
    reply = memory_demo.chat(msg)

print(f"Completed {len(memory_demo.turn_token_usage)} turns.")
print("\nToken usage per turn:")
for t in memory_demo.turn_token_usage:
    print(f"  Turn {t['turn']:2d}: {t['input_tokens']:5d} input tokens, {t['output_tokens']:4d} output tokens")

绘制每轮输入 token 和累计用量。线性和二次增长模式一目了然。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

turns = [t["turn"] for t in memory_demo.turn_token_usage]
input_tokens = [t["input_tokens"] for t in memory_demo.turn_token_usage]
cumulative = np.cumsum(input_tokens)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 左图：每轮输入 token（线性增长）
ax1.bar(turns, input_tokens, color="#6366f1", alpha=0.85)
ax1.set_xlabel("Turn")
ax1.set_ylabel("Input Tokens")
ax1.set_title("Input Tokens per Turn (Linear Growth)")
ax1.set_xticks(turns)

# 右图：累计输入 token（二次增长）
ax2.plot(turns, cumulative, "o-", color="#f43f5e", linewidth=2, markersize=6)
ax2.fill_between(turns, cumulative, alpha=0.15, color="#f43f5e")
ax2.set_xlabel("Turn")
ax2.set_ylabel("Cumulative Input Tokens")
ax2.set_title("Cumulative Token Usage (Quadratic Growth)")
ax2.set_xticks(turns)

plt.tight_layout()
plt.savefig("token_growth.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTotal input tokens across all turns: {int(cumulative[-1]):,}")
print(f"If using a window/summary approach, this could be drastically reduced.")

## 持久化

生产级的缓冲记忆必须在进程重启后存活。数据结构是一个字典列表，所以 JSON 序列化（把它转成文本文件格式）非常直接。


In [ ]:
def save_conversation(memory: ConversationBufferMemory, path: str) -> None:
    """把消息缓冲区持久化到 JSON 文件。"""
    with open(path, "w") as f:
        json.dump(memory.get_history(), f, indent=2, ensure_ascii=False)
    print(f"Saved {len(memory)} messages → {path}")


def load_conversation(path: str, model, **kwargs) -> ConversationBufferMemory:
    """从 JSON 文件恢复一个 ConversationBufferMemory。"""
    with open(path) as f:
        messages = json.load(f)
    mem = ConversationBufferMemory(model=model, **kwargs)
    mem.messages = messages
    print(f"Loaded {len(messages)} messages ← {path}")
    return mem


# 演示存取往返
save_conversation(memory, "conversation.json")
loaded = load_conversation(
    "conversation.json",
    model=llm,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

# 验证加载后的记忆仍可用
reply = loaded.chat("Remind me - what project was I working on?")
print(f"\n🤖 Agent (from loaded memory): {reply}")

## 对比：不带缓冲的 deepagents

deepagents 的 `invoke` 是无状态的。每次调用它只看到你传入的消息。如果不带历史，它每一轮都在「失忆」。上一节的缓冲记忆之所以有效，正是因为我们把历史一直带在身上。下面用同一个智能体直接对比这两种用法。


In [ ]:
naked = create_deep_agent(
    model=llm,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

# 用法一：每次只发最新一条消息（无记忆）
first = naked.invoke({"messages": [{"role": "user", "content": "Hi, my name is Charlie."}]})
print("👤:", "Hi, my name is Charlie.")
print("🤖:", to_text(first["messages"][-1].content), "\n")

second = naked.invoke({"messages": [{"role": "user", "content": "What's my name?"}]})
print("👤:", "What's my name?")
print("🤖:", to_text(second["messages"][-1].content), "  <- 只发了新消息，它不知道你是谁\n")

# 用法二：手动带上完整历史（这正是缓冲记忆在做的事）
print("--- 这次带上完整历史再看 ---")
history = [
    {"role": "user", "content": "Hi, my name is Charlie."},
    {"role": "assistant", "content": to_text(first["messages"][-1].content)},
    {"role": "user", "content": "What's my name?"},
]
third = naked.invoke({"messages": history})
print("🤖:", to_text(third["messages"][-1].content))

**裸 deepagents vs. 加了缓冲记忆的 deepagents：**

| 维度 | 裸 deepagents（每轮只发新消息） | 加缓冲记忆（每轮发完整历史） |
|--------|-------------------|--------------------------------------|
| **跨轮次回忆** | 无，每一轮都「失忆」 | 完整回忆每一个细节 |
| **token 开销** | 每轮常数，最省钱 | 随对话长度线性增长 |
| **实现成本** | 零行代码 | 一个约 50 行的缓冲类 |
| **适合场景** | 单轮问答、无状态任务 | 多轮对话、个性化助手 |

**要点：** deepagents 本身不替你管记忆。对话记忆是你自己加的一层，而最简单的那一层就是缓冲记忆。后续技术（滑动窗口、摘要）都是对这一层的改进。

## 权衡

### 对话缓冲记忆什么时候好用
- **短对话**（20 轮以内），总 token 量宽裕地落在上下文窗口内。
- **原型开发**：先在几分钟内跑起来，再谈优化。
- **要求完美回忆**：任何信息都不会被丢弃或摘要。

### 什么时候会失灵
- **长对话**：每轮输入 token 线性增长，累计成本二次方增长。
- **成本敏感的应用**：每次调用都重发整个历史，代价高昂。
- **延迟敏感的应用**：输入 token 越多，响应越慢。延迟指从发出请求到模型回复之间的等待时间。
- **上下文窗口限制**：缓冲区最终会超过模型上限（依模型不同，从 4k 到百万级 token）。

### 成本示例
假设一场 50 轮的对话，每轮新增约 100 token 的内容：
- 第 1 轮：约 100 输入 token
- 第 50 轮：约 5,000 输入 token
- **50 轮累计输入 token：约 127,500**（如果每轮只发最新一轮，则约 5,000）

### 下一步？
本系列的后续技术解决这些局限：
- **[02：滑动窗口记忆](../02_sliding_window_memory/)** 只保留最近 *k* 条消息。
- **[03：摘要记忆](../03_summary_memory/)** 用 LLM 生成的摘要替换旧轮次。
- **[04：摘要缓冲记忆](../04_summary_buffer_memory/)** 是混合方案：旧消息做摘要，新消息保留原文。
- **[05：token 缓冲记忆](../05_token_buffer_memory/)** 按严格的 token 预算裁剪。


## 延伸阅读

- [deepagents GitHub 仓库](https://github.com/langchain-ai/deepagents)
- [deepagents 官方文档](https://docs.langchain.com/oss/python/deepagents/overview)
- [LangChain ConversationBufferMemory](https://python.langchain.com/docs/modules/memory/types/buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [Lilian Weng, "LLM Powered Autonomous Agents"（Memory 章节）](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)

---

*下一篇：02：滑动窗口记忆 ->*

清理演示过程中产生的临时文件。


In [ ]:
for f in ["conversation.json", "token_growth.png"]:
    if os.path.exists(f):
        os.remove(f)

## 🧪 动手试一试

三个小挑战，加深理解。每个大约需要 10-30 分钟。

### 挑战 1：多人格缓冲
创建两个独立的 `ConversationBufferMemory` 实例，配不同的系统提示词（比如一个家教和一个旅行规划师）。各自跑五轮，对比缓冲区内容如何分化。留意系统提示词对 token 计数的影响。

### 挑战 2：量出成本曲线
把 token 增长演示扩展到 20 轮。记录每轮的 `input_tokens`，按美元计算累计成本（按你所用端点的定价），并打印一张汇总表。把总成本和一个假设只发最近 5 条消息的系统做对比。

### 挑战 3：换上数据库后端
把基于 JSON 的 `save_conversation()` / `load_conversation()` 换成 SQLite 后端。每条消息存一行，列包括角色、内容和时间戳。验证存取往返仍然可用。这会为你理解 21 跨会话记忆里的模式打好基础。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--01-conversation-buffer-memory--conversation-buffer-memory)
